# Inventory Analysis
This notebook analyzes retail inventory data.

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv('retail_store_inventory.csv')
df.head()


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Demand Forecast,Price,Discount,Weather Condition,Holiday/Promotion,Competitor Pricing,Seasonality
0,2022-01-01,S001,P0001,Groceries,North,231,127,55,135.47,33.50,20,Rainy,0,29.69,Autumn
1,2022-01-01,S001,P0002,Toys,South,204,150,66,144.04,63.01,20,Sunny,0,66.16,Autumn
2,2022-01-01,S001,P0003,Toys,West,102,65,51,74.02,27.99,10,Sunny,1,31.32,Summer
3,2022-01-01,S001,P0004,Toys,North,469,61,164,62.18,32.72,10,Cloudy,1,34.74,Autumn
4,2022-01-01,S001,P0005,Electronics,East,166,14,135,9.26,73.64,0,Sunny,0,68.95,Summer


In [2]:
# Format columns for analytical time-series manipulation
df['Date'] = pd.to_datetime(df['Date'])

# Rename columns to avoid syntax issues with spaces
df = df.rename(columns={
    'Inventory Level': 'Inventory',
    'Units Sold': 'Sales',
    'Units Ordered': 'Orders',
    'Demand Forecast': 'Forecast',
    'Weather Condition': 'Weather',
    'Holiday/Promotion': 'Promotion',
    'Competitor Pricing': 'Competitor_Price'
})

print(f"Dataset mapped with {df.shape[0]} rows ready for optimization analysis.")

Dataset mapped with 73100 rows ready for optimization analysis.


In [3]:
# Calculate remaining stock at the end of the business day
df['Ending_Inventory'] = df['Inventory'] - df['Sales']

# Find critical stock-out risks where ending inventory dropped to zero or negative
critical_anomalies = df[df['Ending_Inventory'] <= 0]

print(f"Identified {len(critical_anomalies)} severe stock-out events/risks.")
if len(critical_anomalies) > 0:
    print("\nSample of structural inventory failures:")
    print(critical_anomalies[['Date', 'Store ID', 'Product ID', 'Inventory', 'Sales', 'Ending_Inventory']].head())

Identified 369 severe stock-out events/risks.

Sample of structural inventory failures:
          Date Store ID Product ID  Inventory  Sales  Ending_Inventory
137 2022-01-02     S002      P0018        124    124                 0
301 2022-01-04     S001      P0002         91     91                 0
563 2022-01-06     S004      P0004         50     50                 0
646 2022-01-07     S003      P0007        177    177                 0
802 2022-01-09     S001      P0003        153    153                 0


In [4]:
# Group by Product to find total velocity and demand volatility
sku_summary = df.groupby('Product ID').agg(
    Total_Sales=('Sales', 'sum'),
    Avg_Daily_Sales=('Sales', 'mean'),
    Demand_Volatility=('Sales', 'std')
).reset_index()

# Sort by highest volume to pick our target SKUs
top_skus = sku_summary.sort_values(by='Total_Sales', ascending=False)
print("\nTop 5 High-Volume SKUs for Optimization Focus:")
print(top_skus.head())


Top 5 High-Volume SKUs for Optimization Focus:
   Product ID  Total_Sales  Avg_Daily_Sales  Demand_Volatility
15      P0016       508472       139.116826         109.389823
19      P0020       507708       138.907798         110.229782
13      P0014       507622       138.884268         110.203837
14      P0015       507283       138.791518         109.914956
4       P0005       503648       137.796990         106.854998


In [5]:
# Group by Product ID to find baseline logistics metrics
optimization_df = df.groupby('Product ID').agg(
    Avg_Daily_Sales=('Sales', 'mean'),
    Demand_Volatility=('Sales', 'std')
).reset_index()

# Define our industrial constraints/assumptions
Z_SCORE = 1.65  # 95% Service Level (Industry standard to prevent stock-outs)
LEAD_TIME = 3   # Assume 3 days for a supplier to restock the store

# Apply the mathematical safety stock formula
optimization_df['Safety_Stock'] = (
    Z_SCORE * 
    optimization_df['Demand_Volatility'] * 
    np.sqrt(LEAD_TIME)
)

# Calculate the Reorder Point (ROP) = (Avg Daily Sales * Lead Time) + Safety Stock
optimization_df['Reorder_Point'] = (
    (optimization_df['Avg_Daily_Sales'] * LEAD_TIME) + 
    optimization_df['Safety_Stock']
)

# Let's see our newly engineered metrics for the top 5 items
print("Engineered Safety Stock Framework:")
print(optimization_df.round(2).head())

NameError: name 'np' is not defined